# CineIQ — Offline EvaluationRanking metrics for the hybrid ensemble recommender.**Metrics:**- **Precision@K**: Of the top-K recommended movies, how many did the user actually rate highly?- **Recall@K**: Of all movies the user rated highly, how many appeared in top-K?- **NDCG@K**: Are highly-rated movies ranked higher in the recommendations?**Method:** Hold out the last 20% of each user's ratings (by timestamp) as test set. Recommend on the remaining 80%. Evaluate whether held-out high-rated movies appear in recommendations.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import time
from collections import defaultdict

from src.recommender import (
    load_movies, load_ratings, load_content_index,
    load_svd_model, load_weights, load_popularity,
    content_scores, svd_scores, popularity_scores
)

movies = load_movies()
ratings = load_ratings()
content_index = load_content_index()
svd_model = load_svd_model()
weights = load_weights()
popularity = load_popularity(ratings)

print(f"Movies: {len(movies):,}")
print(f"Ratings: {len(ratings):,}")

## 1. Create Train/Test SplitHold out each user's most recent 20% ratings as test set.

In [ ]:
# Sort by timestamp
ratings_sorted = ratings.sort_values(["userId", "timestamp"])

# For each user, last 20% of ratings go to test
train_list = []
test_list = []

for user_id, group in ratings_sorted.groupby("userId"):
    n = len(group)
    split = int(n * 0.8)
    train_list.append(group.iloc[:split])
    test_list.append(group.iloc[split:])

train = pd.concat(train_list)
test = pd.concat(test_list)

print(f"Train: {len(train):,} ratings")
print(f"Test: {len(test):,} ratings")
print(f"Users in test: {test['userId'].nunique():,}")

## 2. Build User Profiles from Training Data

In [ ]:
# Build user high-rated movie sets from test set (ground truth)
# A movie is "relevant" if user rated it >= 4.0
user_relevant = defaultdict(set)
for _, row in test.iterrows():
    if row["rating"] >= 4.0:
        user_relevant[row["userId"]].add(row["movieId"])

# User training history (for candidate filtering)
user_train_movies = defaultdict(set)
for _, row in train.iterrows():
    user_train_movies[row["userId"]].add(row["movieId"])

print(f"Users with relevant test movies: {len(user_relevant):,}")

## 3. Generate Recommendations for Test Users

In [ ]:
# Sample users for evaluation (full set is slow)
np.random.seed(42)
eval_users = list(user_relevant.keys())
np.random.shuffle(eval_users)
eval_users = eval_users[:500]  # evaluate on 500 users for speed

print(f"Evaluating on {len(eval_users)} users...")

In [ ]:
K = 10
user_recs = {}  # {userId: [movieId, ...]}

t0 = time.time()
for i, user_id in enumerate(eval_users):
    if i % 100 == 0:
        print(f"  {i}/{len(eval_users)}...")

    # Get candidate movies from content-based (top 50)
    # Use any movie the user has rated as seed
    user_movies = list(user_train_movies.get(user_id, set()))
    if not user_movies:
        continue

    # Use highest-rated movie as seed
    user_train_ratings = train[train["userId"] == user_id]
    seed_movie_id = user_train_ratings.loc[user_train_ratings["rating"].idxmax(), "movieId"]
    seed_title = movies[movies["movieId"] == seed_movie_id]["title"].values
    if len(seed_title) == 0:
        continue
    seed_title = seed_title[0]

    # Get candidates from content index
    c_scores = content_scores(seed_title, content_index, movies, n=50)
    if not c_scores:
        continue

    movie_ids = list(c_scores.keys())

    # Normalize content scores
    max_c = max(c_scores.values()) or 1
    c_norm = {mid: v / max_c for mid, v in c_scores.items()}

    # SVD scores
    s_raw = svd_scores(user_id, movie_ids, svd_model)
    max_s = max(s_raw.values()) or 1
    s_norm = {mid: v / max_s for mid, v in s_raw.items()}

    # Popularity
    max_p = max(popularity.values()) if popularity else 1
    p_norm = {mid: popularity.get(mid, 0) / max_p for mid in movie_ids}

    # Ensemble
    final = {}
    for mid in movie_ids:
        final[mid] = (
            weights["w_content"] * c_norm.get(mid, 0)
            + weights["w_svd"] * s_norm.get(mid, 0)
            + weights["w_pop"] * p_norm.get(mid, 0)
        )

    # Remove already-rated movies
    already_rated = user_train_movies.get(user_id, set())
    for mid in already_rated:
        final.pop(mid, None)

    # Top-K
    top_k = sorted(final, key=final.get, reverse=True)[:K]
    user_recs[user_id] = top_k

print(f"Generated recommendations for {len(user_recs)} users in {time.time() - t0:.1f}s")

## 4. Compute Ranking Metrics

In [ ]:
def precision_at_k(recs, relevant, k):
    """Precision@K: fraction of top-K that are relevant."""
    recs_k = recs[:k]
    return len(set(recs_k) & relevant) / k

def recall_at_k(recs, relevant, k):
    """Recall@K: fraction of relevant items found in top-K."""
    recs_k = recs[:k]
    if not relevant:
        return 0.0
    return len(set(recs_k) & relevant) / len(relevant)

def dcg_at_k(recs, relevant, k):
    """DCG@K: discounted cumulative gain."""
    dcg = 0.0
    for i, movie_id in enumerate(recs[:k]):
        if movie_id in relevant:
            dcg += 1.0 / np.log2(i + 2)  # i+2 because log2(1) = 0
    return dcg

def ndcg_at_k(recs, relevant, k):
    """NDCG@K: normalized DCG."""
    dcg = dcg_at_k(recs, relevant, k)
    # Ideal: all relevant items at top
    ideal_recs = list(relevant)[:k]
    idcg = dcg_at_k(ideal_recs, relevant, k)
    if idcg == 0:
        return 0.0
    return dcg / idcg

In [ ]:
# Compute metrics
precisions = []
recalls = []
ndcgs = []

for user_id, recs in user_recs.items():
    relevant = user_relevant.get(user_id, set())
    if not relevant:
        continue

    precisions.append(precision_at_k(recs, relevant, K))
    recalls.append(recall_at_k(recs, relevant, K))
    ndcgs.append(ndcg_at_k(recs, relevant, K))

print(f"=" * 50)
print(f"CineIQ Hybrid Ensemble — Offline Evaluation")
print(f"=" * 50)
print(f"Users evaluated: {len(precisions)}")
print(f"")
print(f"Precision@{K}: {np.mean(precisions):.4f} (+/- {np.std(precisions):.4f})")
print(f"Recall@{K}:    {np.mean(recalls):.4f} (+/- {np.std(recalls):.4f})")
print(f"NDCG@{K}:      {np.mean(ndcgs):.4f} (+/- {np.std(ndcgs):.4f})")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(precisions, bins=20, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_title(f"Precision@{K} Distribution")
axes[0].set_xlabel("Precision")
axes[0].axvline(np.mean(precisions), color="red", linestyle="--", label=f'Mean: {np.mean(precisions):.4f}')
axes[0].legend()

axes[1].hist(recalls, bins=20, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_title(f"Recall@{K} Distribution")
axes[1].set_xlabel("Recall")
axes[1].axvline(np.mean(recalls), color="red", linestyle="--", label=f'Mean: {np.mean(recalls):.4f}')
axes[1].legend()

axes[2].hist(ndcgs, bins=20, edgecolor="black", alpha=0.7, color="mediumseagreen")
axes[2].set_title(f"NDCG@{K} Distribution")
axes[2].set_xlabel("NDCG")
axes[2].axvline(np.mean(ndcgs), color="red", linestyle="--", label=f'Mean: {np.mean(ndcgs):.4f}')
axes[2].legend()

plt.tight_layout()
plt.show()

## 5. Comparison: Ensemble vs Individual Components

In [ ]:
# Compare ensemble vs content-only vs SVD-only
# (Using same user set, same K)

content_only_recs = {}
svd_only_recs = {}

for user_id in eval_users[:100]:  # subset for speed
    user_train_ratings = train[train["userId"] == user_id]
    if len(user_train_ratings) == 0:
        continue
    seed_movie_id = user_train_ratings.loc[user_train_ratings["rating"].idxmax(), "movieId"]
    seed_title = movies[movies["movieId"] == seed_movie_id]["title"].values
    if len(seed_title) == 0:
        continue
    seed_title = seed_title[0]

    # Content only
    c_scores = content_scores(seed_title, content_index, movies, n=50)
    if c_scores:
        top_content = sorted(c_scores, key=c_scores.get, reverse=True)[:K]
        content_only_recs[user_id] = top_content

    # SVD only (rank by SVD score for user)
    c_ids = list(c_scores.keys()) if c_scores else []
    if c_ids:
        s_raw = svd_scores(user_id, c_ids, svd_model)
        top_svd = sorted(s_raw, key=s_raw.get, reverse=True)[:K]
        svd_only_recs[user_id] = top_svd

# Compute metrics for each
def eval_method(recs_dict, name):
    p, r, n = [], [], []
    for uid, recs in recs_dict.items():
        rel = user_relevant.get(uid, set())
        if not rel:
            continue
        p.append(precision_at_k(recs, rel, K))
        r.append(recall_at_k(recs, rel, K))
        n.append(ndcg_at_k(recs, rel, K))
    if p:
        print(f"{name:25s}  P@{K}={np.mean(p):.4f}  R@{K}={np.mean(r):.4f}  NDCG@{K}={np.mean(n):.4f}")

print(f"=" * 70)
print(f"Component Comparison (K={K})")
print(f"=" * 70)
eval_method(content_only_recs, "Content-Based Only")
eval_method(svd_only_recs, "SVD Only")
eval_method(user_recs, "Hybrid Ensemble")